In [ ]:
# nothing
print("test")

In [ ]:
import os
os.environ["HF_HOME"] = "/data/cat/ws/thha024h-thomas_internship/huggingface"

#print(f"Huggingface cache set to: {os.environ["HF_HOME"]}")

In [ ]:
import json
import re
import pandas as pd

from vllm import LLM, SamplingParams

data_path = "./"

In [ ]:
dd = pd.read_parquet(os.path.join(data_path, "dd.parquet"))

In [ ]:
# read prompt file
with open("prompt8.md", "r", encoding="utf-8") as f:
  prompt_system = f.read()

# LLM Coding
## Process Posts with LLM

In [ ]:
temp_sample = dd[["id", "xid", "text"]].reset_index(drop=True)
temp_sample.tail()

In [ ]:
# SETUP
MODEL_NAME = "/data/cat/ws/thha024h-thomas_internship/huggingface/Qwen3-235B-A22B-Instruct-2507-FP8"
MODEL_NAME = "/data/cat/ws/thha024h-thomas_internship/huggingface/Qwen2.5-3B-Instruct"
MODEL_NAME = "/data/cat/ws/thha024h-thomas_internship/huggingface/gpt-oss-120b"

llm_kwargs = dict(
    model = MODEL_NAME
)

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=4048
)

llm = LLM(**llm_kwargs)
tokenizer = llm.get_tokenizer()

temp_sample = dd.sample(n=3)[["id", "xid", "text"]].reset_index(drop=True)
temp_sample = dd[["id", "xid", "text"]].reset_index(drop=True)
# PROCESS TEXT
prompts_list = []
rows_list = []

for row in temp_sample.itertuples():

    conversation = [
        {"role": "system", "content": prompt_system},
        {"role": "user", "content": row.text}
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(
        conversation, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    prompts_list.append(formatted_prompt)
    rows_list.append(row)


print(f"Starting inference on {len(prompts_list)} items...")
outputs = llm.generate(prompts_list, sampling_params)

## Process Results

In [ ]:
d_results = []

for row, output in zip(rows_list, outputs):
    answer = output.outputs[0].text
    
    # --- 1. Attempt Standard JSON Parsing ---
    try:
        # Isolate the JSON object (find first { and last })
        json_str = answer[answer.find("{"):answer.rfind("}") + 1]
        answer_data = json.loads(json_str)
        
    except (json.JSONDecodeError, ValueError, AttributeError):
        # --- 2. Regex Fallback (Fixes "Bad Quotes" inside text) ---
        # The model often breaks JSON by putting double quotes inside strings.
        # We manually extract the fields based on your specific keys.
        try:
            parsed = {}
            
            # A. Extract Text Fields
            # Captures text between: "key": " ...AND... ", (comma) or "} (end brace)
            str_keys = [
                "holistic_redescription", 
                "social_actors_analysis", 
                "populist_explanation", 
                "elitist_explanation", 
                "intensity_explanation"
            ]
            
            for key in str_keys:
                match = re.search(f'"{key}"\s*:\s*"(.*?)"\s*(?:,|}}\s*$)', answer, re.DOTALL)
                if match:
                    # Clean up: Replace internal double quotes with single quotes to be safe
                    parsed[key] = match.group(1).replace('"', "'")
                else:
                    parsed[key] = None

            # B. Extract Number Fields
            num_keys = ["populist_score", "elitist_score", "intensity_score"]
            for key in num_keys:
                match = re.search(f'"{key}"\s*:\s*(-?\d+)', answer)
                if match:
                    parsed[key] = int(match.group(1))
                else:
                    parsed[key] = None
            
            answer_data = parsed
            
        except Exception as regex_error:
            # If both fail, log it
            answer_data = {"error": f"Parse Failed: {regex_error}", "raw_output": answer}

    # --- 3. Append to Results ---
    d_results.append({
        "id": row.id, 
        "xid": row.xid, 
        "text": row.text,
        **answer_data
    })

# Convert to DataFrame
d = pd.DataFrame(d_results)

In [ ]:
d.head()

# Save Results

In [ ]:
d.to_parquet(f'{data_path}/llm_coding2.parquet')